# Learning Fusion 3 — Hands-On: Early / Mid / Late Fusion

This notebook lets you **see the numbers and tensor shapes** behind **Section 3** of
`LEARNING_FUSION.md` / `LEARNING_FUSION.html` — the three places you can fuse camera + LiDAR.

It is **self-contained** — no uploads needed — so it runs as-is in **Google Colab**
(which ships with `numpy`, `matplotlib`, `torch`, `torchvision`).

The code mirrors your own repo (faithful interfaces, simplified internals so it runs in
seconds on a CPU):
- `EarlyFusion2D`  ↔  `fusion/fusion/early_2d/model.py`   (fuse at the **input**)
- `MidFusion2D`    ↔  `fusion/fusion/mid_2d/model.py`     (fuse at the **feature grid**)
- `LateFusion2D`   ↔  `fusion/fusion/late_2d/model.py`    (fuse at the **decision** / box list)
- backbone / head / encode / decode / IoU / NMS  ↔  `common/backbones`, `fusion/heads.py`,
  `fusion/common_2d.py`, `common/geometry/boxes2d.py`, `train/losses_2d.py`

**Convention (KITTI):** camera frame — **x right, y down, z forward**; LiDAR (velo) frame —
**x forward, y left, z up**. Feature grid stride = 16, so a 384×1280 image → a **24×80** grid.

> Run top-to-bottom once. Each variant: build the input → run the real forward pass (shapes
> printed at every step) → train a few steps so a box actually appears → decode. Then the
> late-fusion section walks the IoU / merge / NMS math one line at a time.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0); np.random.seed(0)

# ---- config: mirrors config/base.yaml (+ per-variant yamls), shrunk for a fast CPU demo ----
cfg = dict(
    image_size=(384, 1280),     # [H, W]  (KITTI aspect)  -> grid 24 x 80 at stride 16
    stride=16,
    num_classes=1,
    classes=["Car"],
    fusion_channels=32,        # base.yaml uses 256; shrunk so a CPU forward is instant
    lidar_feat_channels=16,    # base.yaml uses 32
    head_hidden=64,            # heads.py uses 128
    merge_iou_thresh=0.5,      # late_2d.yaml
    merge_conf_mode="mean",    # late_2d.yaml  (mean | max)
    max_points=15000,
)
H, W = cfg["image_size"]; s = cfg["stride"]
Hg, Wg = H // s, W // s
print(f"image {H}x{W}  ->  feature grid {Hg}x{Wg} at stride {s}")
print(f"fusion_channels={cfg['fusion_channels']}  lidar_feat_channels={cfg['lidar_feat_channels']}")


## Section 3 · Where does fusion happen?

A detector pipeline is roughly:

```
   sensor raw  ->  [preprocess]  ->  [backbone]  ->  [features]  ->  [head]  ->  [boxes]
     camera : RGB image ..............................................->  2D boxes
     lidar  : point cloud ............................................->  2D boxes
```

The three variants differ only in **where the two streams meet**:

| variant | fusion point | what is combined | export story |
|--------|--------------|------------------|--------------|
| **Early** (data)    | input            | RGB + projected depth  ->  **4-channel image** | clean: net is just a 4-ch detector |
| **Mid** (feature)   | feature grid     | `F_cam` + `F_lid` **aligned** -> concat + 1×1 conv | the modern sweet spot |
| **Late** (decision) | box list         | two independent detectors -> **IoU merge + NMS** | needs two engines + a C++ merge |

> Early = combine the *raw data*; Mid = combine the *learned features*; Late = combine the
> *outputs*. The further down the pipeline you fuse, the more each sensor is a self-contained
> expert and the less the two streams can correct each other — but the easier the deploy.


In [ ]:
# ---- Calib: mirrors fusion/common/sensors/calibration.py (numpy + a torch_matrices hook) ----
class Calib:
    """P2 (3x4), R0_rect (3x3), Tr_velo_to_cam (3x4) as homogeneous 4x4. cam: x-right,y-down,z-fwd."""
    def __init__(self, P2, R0_rect, Tr_velo_to_cam):
        self.P2 = np.asarray(P2, float).reshape(3, 4)
        self.R0 = np.eye(4); self.R0[:3, :3] = np.asarray(R0_rect, float).reshape(3, 3)
        self.V2C = np.eye(4); self.V2C[:3, :] = np.asarray(Tr_velo_to_cam, float).reshape(3, 4)
        self.P  = np.eye(4); self.P[:3, :] = self.P2

    def velo_to_cam(self, pts):
        pts = np.atleast_2d(np.asarray(pts, float)); n = pts.shape[0]
        h = np.hstack([pts[:, :3], np.ones((n, 1))])
        return (self.R0 @ self.V2C @ h.T).T[:, :3]

    def cam_to_image(self, pts_cam):
        pts_cam = np.atleast_2d(np.asarray(pts_cam, float)); n = pts_cam.shape[0]
        h = np.hstack([pts_cam, np.ones((n, 1))])
        img = (self.P @ h.T).T[:, :3]
        depth = img[:, 2]
        uv = img[:, :2] / np.where(depth == 0, 1e-9, depth)[:, None]
        return uv, depth

    def velo_to_image(self, pts):
        cam = self.velo_to_cam(pts); uv, depth = self.cam_to_image(cam)
        return uv, depth, cam

    def torch_matrices(self, device, dtype):
        return {"R0": torch.tensor(self.R0, device=device, dtype=dtype),
                "V2C": torch.tensor(self.V2C, device=device, dtype=dtype),
                "P":  torch.tensor(self.P,  device=device, dtype=dtype)}

# ---- projection helpers: mirror common/sensors/projection.py (numpy) ----
def lidar_to_image(points, calib, H, W):
    uv, depth, _ = calib.velo_to_image(points[:, :3])
    valid = (depth > 0) & (uv[:, 0] >= 0) & (uv[:, 0] < W) & (uv[:, 1] >= 0) & (uv[:, 1] < H)
    return uv, depth, valid

def render_depth_image(points, calib, H, W, fill=0.0):
    uv, depth, valid = lidar_to_image(points, calib, H, W)
    img = np.full((H, W), fill, np.float32)
    uv, depth = uv[valid], depth[valid]
    if uv.shape[0] == 0: return img
    u, v = uv[:, 0].astype(int), uv[:, 1].astype(int)
    order = np.argsort(-depth)        # far first -> near overwrites (closest surface wins)
    u, v, depth = u[order], v[order], depth[order]
    img[v, u] = depth.astype(np.float32)
    return img

def render_lidar_3ch_image(points, calib, H, W):
    """(H,W,3) = [depth, height(z), intensity] -- the lidar-only-2D baseline input."""
    uv, depth, valid = lidar_to_image(points, calib, H, W)
    img = np.zeros((H, W, 3), np.float32)
    uv, depth = uv[valid], depth[valid]
    if uv.shape[0] == 0: return img
    u, v = uv[:, 0].astype(int), uv[:, 1].astype(int)
    hgt = points[valid, 2].astype(np.float32)
    inten = points[valid, 3].astype(np.float32) if points.shape[1] >= 4 else np.zeros_like(hgt)
    order = np.argsort(-depth)
    u, v, depth, hgt, inten = u[order], v[order], depth[order], hgt[order], inten[order]
    img[v, u, 0] = depth; img[v, u, 1] = hgt; img[v, u, 2] = inten
    return img

print("Calib + projection helpers ready (mirror common/sensors/calibration.py & projection.py)")


In [ ]:
# ---- build a calibration (self-consistent intrinsics for the 384x1280 demo image) ----
fx = fy = 720.0
cx, cy = W / 2., H / 2.            # 640.0, 192.0
P2 = np.array([[fx, 0, cx, 0], [0, fy, cy, 0], [0, 0, 1, 0]], float)
# velo (x-fwd,y-left,z-up) -> cam (x-right,y-down,z-fwd):  cam_x=-velo_y, cam_y=-velo_z, cam_z=velo_x
R_velo2cam = np.array([[0, -1, 0], [0, 0, -1], [1, 0, 0]], float)
t = np.array([0.0, -0.08, -0.27])             # LiDAR ~27cm behind, 8cm above the camera
Tr = np.hstack([R_velo2cam, t.reshape(3, 1)])
R0 = np.eye(3)
calib = Calib(P2, R0, Tr)

# ---- synthetic scene: a car (box of surface points) + sparse ground, in velo frame ----
def make_car_pts(center, dims=(4.0, 2.0, 1.5), n_per_face=40, intensity=0.55):
    cx_, cy_, cz_ = center; lx, ly, lz = dims
    xs = [-lx/2, lx/2]; ys = [-ly/2, ly/2]; zs = [-lz/2, lz/2]
    P = []
    for (a, ax) in [(xs, 0), (ys, 1), (zs, 2)]:
        for v in a:
            for _ in range(n_per_face):
                p = [cx_, cy_, cz_]
                p[ax] += v
                for k in range(3):
                    if k != ax: p[k] += np.random.uniform(-dims[k]/2, dims[k]/2)
                P.append(p + [intensity])
    return np.array(P, float)

car_pts  = make_car_pts([10.0, 2.0, 1.0])                     # ~10 m ahead, 2 m left
ground   = np.column_stack([np.random.uniform(5, 30, 120),   # ground plane z~0
                            np.random.uniform(-8, 8, 120),
                            np.full(120, 0.02),
                            np.full(120, 0.1)])
points = np.vstack([car_pts, ground])                         # (N,4): x,y,z,intensity
np.random.shuffle(points)

# ---- ground-truth 2D box = projection of the car's 8 corners ----
corners = np.array([[x, y, z] for x in (8, 12) for y in (1, 3) for z in (0.25, 1.75)], float)
uv_c, dep_c, _ = calib.velo_to_image(corners)
infront = dep_c > 0
gt_box = np.array([uv_c[infront, 0].min(), uv_c[infront, 1].min(),
                   uv_c[infront, 0].max(), uv_c[infront, 1].max()])
gt_label = np.array([0], int)

# ---- a synthetic RGB image: gradient + a brighter patch where the car projects ----
img = np.zeros((H, W, 3), np.float32)
img[..., 0] = np.linspace(0.05, 0.25, W)[None, :]            # faint horizon gradient
img += np.random.uniform(-0.02, 0.02, img.shape).astype(np.float32)
x1, y1, x2, y2 = gt_box.astype(int)
img[max(0, y1):y2, max(0, x1):x2] += 0.35                    # car patch is brighter

np.set_printoptions(precision=3, suppress=True)
print(f"points: {points.shape}  (car {car_pts.shape[0]} + ground {ground.shape[0]})")
print(f"intensity range: [{points[:,3].min():.2f}, {points[:,3].max():.2f}]")
print(f"GT 2D box (x1,y1,x2,y2) = {gt_box}   label = {cfg['classes'][gt_label[0]]}")
print(f"car center velo [10,2,1] -> cam {calib.velo_to_cam([[10,2,1]])[0]} "
      f"-> pixel {calib.velo_to_image([[10,2,1]])[0][0].round(1)}")


### The scene, visualized

Left: the **depth image** rendered by projecting every LiDAR point to its pixel (closest
surface wins per pixel). Right: every projected point scattered on the image plane, with the
**GT box** drawn from the car's projected corners. This is the data all three variants start from.


In [ ]:
depth_img = render_depth_image(points, calib, H, W)
uv, dep, valid = lidar_to_image(points, calib, H, W)

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
im = ax[0].imshow(depth_img, cmap="viridis"); ax[0].set_title(f"LiDAR depth image ({H}x{W})\n"
                 f"covered pixels: {(depth_img>0).sum()} ({100*(depth_img>0).mean():.1f}% of frame)")
plt.colorbar(im, ax=ax[0], fraction=0.025, label="depth (m)")
sc = ax[1].scatter(uv[valid, 0], uv[valid, 1], c=dep[valid], s=4, cmap="viridis")
ax[1].add_patch(patches.Rectangle((gt_box[0], gt_box[1]), gt_box[2]-gt_box[0], gt_box[3]-gt_box[1],
                                  ec="red", fc="none", lw=2))
ax[1].set_title("projected points + GT box (red)\n(car cluster center ~ pixel 492,112)")
ax[1].set_xlim(0, W); ax[1].set_ylim(H, 0); ax[1].set_aspect("equal")
plt.colorbar(sc, ax=ax[1], fraction=0.025, label="depth (m)")
plt.tight_layout(); plt.show()
print(f"depth range (covered pixels): [{depth_img[depth_img>0].min():.2f}, "
      f"{depth_img[depth_img>0].max():.2f}] m")


## 3.1 · Early fusion (data-level)

Combine the **raw data**: render LiDAR as a single depth image, normalize it, and append it as a
**4th channel** to RGB. The backbone's first conv accepts 4 channels; everything downstream is a
normal detector. The fusion *is* the input — there is no separate fusion op in the network.

```
RGB (B,3,H,W)  ┐
               ├─ cat ─> (B,4,H,W) ─> backbone ─> feat (B,C,24,80) ─> head ─> boxes
depth (B,1,H,W)┘
```
This is exactly `EarlyFusion2D._build_input` in `fusion/early_2d/model.py`.


In [ ]:
# ---- build the 4-channel input step by step (mirror EarlyFusion2D._build_input) ----
img_t = torch.from_numpy(img).permute(2, 0, 1).unsqueeze(0)          # (1,3,H,W) 0..1
depth_t = torch.from_numpy(render_depth_image(points, calib, H, W))  # (H,W) in metres
depth_norm = (depth_t / 80.0).clamp(0, 1).unsqueeze(0).unsqueeze(0)  # (1,1,H,W) ~[0,1]

print("step 1  RGB      :", tuple(img_t.shape),   " range [%.3f, %.3f]" % (img_t.min(), img_t.max()))
print("step 2  depth(m) :", tuple(depth_t.shape), " range [%.2f, %.2f]  (raw metres)" %
      (depth_t[depth_t>0].min() if (depth_t>0).any() else 0,
       depth_t.max()))
print("step 3  depth/80 :", tuple(depth_norm.shape), " range [%.3f, %.3f]  (normalized ~[0,1])" %
      (depth_norm.min(), depth_norm.max()))

early_input = torch.cat([img_t, depth_norm], dim=1)                 # (1,4,H,W)
print("step 4  cat dim=1:", tuple(early_input.shape),
      " channels = [R, G, B, depth]")
print("          per-channel mean:", early_input.mean(dim=(2,3)).tolist())
print(f"          depth channel coverage: {100*(depth_t>0).float().mean():.1f}% of pixels carry a return")


In [ ]:
fig, ax = plt.subplots(1, 4, figsize=(14, 3))
for i, (name, cmap) in enumerate(zip(["R","G","B","depth"],
                                     ["Reds","Greens","Blues","viridis"])):
    ax[i].imshow(early_input[0, i].numpy(), cmap=cmap); ax[i].set_title(name); ax[i].axis("off")
plt.suptitle("The 4-channel early-fusion input (each channel is an (H,W) map)")
plt.tight_layout(); plt.show()


### Shared network blocks (used by all three variants)

To keep the CPU demo instant, the backbone below is a **tiny stride-16 conv stack** that
returns the same `{stride: feat}` dict interface as the repo's
`ImageBackbone` (ResNet-18 + FPN, strides {4,8,16,32}). We only use the **stride-16** level, which
matches `cfg["stride"]=16`. The head, target encoder, decoder, and losses are faithful copies of
`fusion/heads.py`, `common/geometry/boxes2d.py`, `fusion/common_2d.py`, `train/losses_2d.py`.


In [ ]:
# ---- TinyImageBackbone: same {stride: feat} interface as common/backbones/image_backbone.py ----
class TinyImageBackbone(nn.Module):
    def __init__(self, in_channels=3, out_channels=32):
        super().__init__()
        self.strides = [16]
        self.net = nn.Sequential(
            nn.Conv2d(in_channels, 16, 3, stride=2, padding=1), nn.BatchNorm2d(16), nn.ReLU(inplace=True),
            nn.Conv2d(16, 24, 3, stride=2, padding=1), nn.BatchNorm2d(24), nn.ReLU(inplace=True),
            nn.Conv2d(24, 32, 3, stride=2, padding=1), nn.BatchNorm2d(32), nn.ReLU(inplace=True),
            nn.Conv2d(32, out_channels, 3, stride=2, padding=1), nn.BatchNorm2d(out_channels), nn.ReLU(inplace=True),
        )
        self.out_channels = out_channels
    def forward(self, x):
        return {str(self.strides[0]): self.net(x)}

# ---- CenterHead2D: faithful copy of fusion/heads.py ----
class CenterHead2D(nn.Module):
    def __init__(self, in_channels, num_classes=1, hidden=64):
        super().__init__()
        self.head = nn.Sequential(nn.Conv2d(in_channels, hidden, 3, padding=1), nn.ReLU(inplace=True),
                                  nn.Conv2d(hidden, hidden, 3, padding=1), nn.ReLU(inplace=True))
        self.heat = nn.Conv2d(hidden, num_classes, 1)
        self.off  = nn.Conv2d(hidden, 2, 1)
        self.size = nn.Conv2d(hidden, 2, 1)
        self.heat.bias.data.fill_(-2.19)          # sigmoid(-2.19) ~ 0.1 baseline
    def forward(self, feat):
        h = self.head(feat)
        return {"heat": self.heat(h), "off": self.off(h), "size": self.size(h)}

# ---- encode/decode + losses: faithful copies of boxes2d.py & losses_2d.py ----
def gaussian_radius(h, w, min_overlap=0.7, min_radius=2):
    # NOTE: min_radius floors the radius so a single-box demo gets a gaussian *blob*
    # (not a point). CenterNet's exact formula yields radius=0 for large boxes, which
    # leaves focal loss with 1 positive cell vs ~1920 negatives -> the net learns
    # 'predict zero everywhere'. The repo's boxes2d.py keeps min_radius=0 because real
    # training has many boxes/frames (the single-point case is rare there).
    b1 = h + w; a2 = 4; b2 = 2*(h+w)+1; c2 = 4*min_overlap
    dc = b1**2 - 4*a2*(c2 + w*h); dr = b2**2 - 16*c2*w*h
    rc = (b1 + np.sqrt(dc))/(2*a2) if dc > 0 else 0.0
    rr = (b2 + np.sqrt(dr))/(2*a2) if dr > 0 else 0.0
    return max(min_radius, int(min(rc, rr)))

def _draw_gaussian(heatmap, center, radius):
    diameter = 2*radius+1; sigma = diameter/6.0
    g = torch.tensor(np.exp(-((np.arange(diameter)-radius)**2)/(2*sigma**2)),
                     dtype=heatmap.dtype, device=heatmap.device)
    g2d = torch.outer(g, g); cy, cx = int(center[0]), int(center[1])
    Hh, Ww = heatmap.shape[-2:]; r = radius
    y0, y1 = max(0, cy-r), min(Hh, cy+r+1); x0, x1 = max(0, cx-r), min(Ww, cx+r+1)
    gy0, gx0 = y0-(cy-r), x0-(cx-r)
    if y1>y0 and x1>x0:
        heatmap[..., y0:y1, x0:x1] = torch.maximum(
            heatmap[..., y0:y1, x0:x1], g2d[gy0:gy0+(y1-y0), gx0:gx0+(x1-x0)])

def encode_boxes2d(boxes, labels, Hg, Wg, stride, num_classes, device):
    heat = torch.zeros((num_classes, Hg, Wg), device=device)
    off = torch.zeros((2, Hg, Wg), device=device); size = torch.zeros((2, Hg, Wg), device=device)
    if boxes.shape[0] == 0: return {"heat": heat, "off": off, "size": size}
    w = (boxes[:, 2]-boxes[:, 0]).clamp(min=1); h = (boxes[:, 3]-boxes[:, 1]).clamp(min=1)
    cx = (boxes[:, 0]+boxes[:, 2])/2; cy = (boxes[:, 1]+boxes[:, 3])/2
    gcy = (cy/stride).long().clamp(0, Hg-1); gcx = (cx/stride).long().clamp(0, Wg-1)
    for i in range(boxes.shape[0]):
        _draw_gaussian(heat[int(labels[i])], (gcy[i].item(), gcx[i].item()),
                       gaussian_radius(h[i].item(), w[i].item()))
    off[0, gcy, gcx] = (cy/stride) - gcy.float(); off[1, gcy, gcx] = (cx/stride) - gcx.float()
    size[0, gcy, gcx] = h; size[1, gcy, gcx] = w
    return {"heat": heat, "off": off, "size": size}

def build_target_2d(boxes, labels, cfg, device):
    return {k: v.unsqueeze(0) for k, v in encode_boxes2d(
        boxes.to(device), labels.to(device), Hg, Wg, cfg["stride"], cfg["num_classes"], device).items()}

def _topk_heatmap(heat, k=40):
    B, C, Hh, Ww = heat.shape; heat = heat.view(B, C, -1)
    scores, idx = heat.topk(k, dim=-1); scores, idx = scores.view(B, -1), idx.view(B, -1)
    cls = (idx // (Hh*Ww)).long(); cell = idx % (Hh*Ww)
    return scores, cls, (cell // Ww).long(), (cell % Ww).long()

def decode_boxes2d(pred, stride, k=40, thresh=0.15):
    heat = pred["heat"].sigmoid(); off = pred["off"]; size = pred["size"]
    B, C, Hgh, Wgh = heat.shape
    scores, labels, ys, xs = _topk_heatmap(heat, k)
    off = off.permute(0, 2, 3, 1).contiguous().view(B, -1, 2)
    size = size.permute(0, 2, 3, 1).contiguous().view(B, -1, 2)
    idx = ys*Wgh + xs
    off_sel = torch.gather(off, 1, idx.unsqueeze(-1).expand(-1, -1, 2))
    size_sel = torch.gather(size, 1, idx.unsqueeze(-1).expand(-1, -1, 2))
    cy = (ys.float()+off_sel[..., 0])*stride; cx = (xs.float()+off_sel[..., 1])*stride
    h = size_sel[..., 0]; w = size_sel[..., 1]; out = []
    for b in range(B):
        m = scores[b] > thresh; s = scores[b][m]
        x1 = cx[b][m]-w[b][m]/2; y1 = cy[b][m]-h[b][m]/2; x2 = cx[b][m]+w[b][m]/2; y2 = cy[b][m]+h[b][m]/2
        out.append({"boxes": torch.stack([x1, y1, x2, y2], dim=-1),
                    "scores": s, "labels": labels[b][m]})
    return out

def focal_loss(pred, target, alpha=2.0, beta=4.0, eps=1e-6):
    pred = pred.sigmoid().clamp(eps, 1-eps)
    pos = target.eq(1).float(); neg = target.lt(1).float()
    num_pos = pos.sum().clamp(min=1)
    return (-((1-pred)**alpha)*torch.log(pred)*pos
            -((1-target)**beta)*(pred**alpha)*torch.log(1-pred)*neg).sum()/num_pos

def l1_loss(pred, target, mask):
    return F.l1_loss(pred*mask, target*mask, reduction="sum")/mask.sum().clamp(min=1)

def compute_2d_loss(pred, target):
    mask = (target["heat"].max(dim=1, keepdim=True)[0] > 0.99).float()
    # size is scaled down by stride so its L1 (~150px) doesn't drown the focal term (~2);
    # the repo's losses_2d.py keeps raw size because the real backbone can absorb it.
    s = cfg["stride"]
    return (focal_loss(pred["heat"], target["heat"])
            + l1_loss(pred["off"], target["off"], mask)
            + 0.1*l1_loss(pred["size"]/s, target["size"]/s, mask))

# ---- IoU + NMS (numpy): faithful copies of boxes2d.py (used by the late merge) ----
def iou2d(a, b):
    a = np.asarray(a, float); b = np.asarray(b, float)
    if a.shape[0] == 0 or b.shape[0] == 0: return np.zeros((a.shape[0], b.shape[0]))
    xa1, ya1, xa2, ya2 = a[:, 0:1], a[:, 1:2], a[:, 2:3], a[:, 3:4]
    xb1, yb1, xb2, yb2 = b[:, 0], b[:, 1], b[:, 2], b[:, 3]
    iw = (np.minimum(xa2, xb2)-np.maximum(xa1, xb1)).clip(min=0)
    ih = (np.minimum(ya2, yb2)-np.maximum(ya1, yb1)).clip(min=0)
    inter = iw*ih; area = (xa2-xa1)*(ya2-ya1) + (xb2-xb1)*(yb2-yb1) - inter
    return inter/(area+1e-9)

def nms2d(boxes, scores, iou_thresh=0.45):
    boxes = np.asarray(boxes, float); scores = np.asarray(scores, float)
    if boxes.shape[0] == 0: return np.array([], int)
    order = scores.argsort()[::-1]; keep = []
    while order.size > 0:
        i = order[0]; keep.append(i)
        if order.size == 1: break
        ovs = iou2d(boxes[i:i+1], boxes[order[1:]])[0]
        order = order[1:][ovs < iou_thresh]
    return np.array(keep, int)

device = "cpu"
print("shared blocks ready: TinyImageBackbone, CenterHead2D, encode/decode, losses, IoU, NMS")


### Early-fusion forward pass — shapes at every step

Now we run the real forward. Watch the tensor shrink from `(1, 4, 384, 1280)` down to a
`(1, 32, 24, 80)` feature grid, then the head turns that into three maps:
`heat (1,1,24,80)`, `off (1,2,24,80)`, `size (1,2,24,80)` — exactly like the repo's `CenterHead2D`.


In [ ]:
torch.manual_seed(0)
early_backbone = TinyImageBackbone(in_channels=4, out_channels=cfg["fusion_channels"])
early_head = CenterHead2D(cfg["fusion_channels"], cfg["num_classes"], cfg["head_hidden"])

x = early_input
print("input  x        :", tuple(x.shape), "  <- (B,4,H,W)  RGB + depth")
feat = early_backbone(x)["16"]
print("backbone(x)['16']:", tuple(feat.shape), "  <- (B,C,Hg,Wg)  384/16=24, 1280/16=80")
pred = early_head(feat)
for k, v in pred.items():
    print(f"head -> {k:5s}    :", tuple(v.shape))
print()
print(f"raw heat logits range: [{pred['heat'].min():.3f}, {pred['heat'].max():.3f}]  "
      f"(untrained; bias=-2.19 -> sigmoid ~ {torch.sigmoid(pred['heat']).mean():.3f})")


In [ ]:
# ---- train a few steps so a real box appears, then decode ----
target = build_target_2d(torch.from_numpy(gt_box[None]).float(),
                         torch.from_numpy(gt_label[None]), cfg, device)
opt = torch.optim.Adam(list(early_backbone.parameters()) + list(early_head.parameters()), lr=3e-3)
print("step |  loss   | heat  | off   | size")
hist = []
for step in range(1, 121):
    opt.zero_grad()
    p = early_head(early_backbone(early_input)["16"])
    loss = compute_2d_loss(p, target)
    loss.backward(); opt.step()
    if step % 30 == 0 or step == 1:
        with torch.no_grad():
            mask = (target["heat"].max(dim=1, keepdim=True)[0] > 0.99).float()
            h = focal_loss(p["heat"], target["heat"]).item()
            o = l1_loss(p["off"], target["off"], mask).item()
            sz = l1_loss(p["size"]/cfg["stride"], target["size"]/cfg["stride"], mask).item()
        print(f"{step:4d} | {loss.item():.4f} | {h:.3f} | {o:.3f} | {sz:.3f}")
        hist.append((step, loss.item()))

with torch.no_grad():
    dec = decode_boxes2d(early_head(early_backbone(early_input)["16"]), cfg["stride"], k=10, thresh=0.15)[0]
print("\nGT box     :", gt_box.round(1))
if dec["boxes"].shape[0]:
    bx = dec["boxes"][0].numpy().round(1); print("decoded box :", bx, " score=%.3f" % dec["scores"][0])
else:
    print("decoded box : (none above threshold)")


### Early-fusion: what the network learned

After ~120 steps the heatmap peaks at the car's grid cell. Below: the learned heatmap and the
decoded box drawn over the 4-channel input (depth channel shown).


In [ ]:
with torch.no_grad():
    hm = early_head(early_backbone(early_input)["16"])["heat"][0, 0].sigmoid().numpy()
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].imshow(hm, cmap="hot"); ax[0].set_title("learned heatmap (24x80 grid)")
ax[0].set_aspect("equal")
ax[1].imshow(early_input[0, 3].numpy(), cmap="viridis")
ax[1].add_patch(patches.Rectangle((gt_box[0], gt_box[1]), gt_box[2]-gt_box[0], gt_box[3]-gt_box[1],
                                  ec="lime", fc="none", lw=2, label="GT"))
if dec["boxes"].shape[0]:
    b = dec["boxes"][0].numpy()
    ax[1].add_patch(patches.Rectangle((b[0], b[1]), b[2]-b[0], b[3]-b[1], ec="red", fc="none",
                                      lw=2, ls="--", label="decoded"))
ax[1].set_title("depth channel + GT(lime) / decoded(red)"); ax[1].legend(loc="upper right")
plt.tight_layout(); plt.show()


### Early-fusion blind modes

`EarlyFusion2D` has two deploy-time robustness switches (`fusion/base.py`): `cam_blind` zeros the
RGB, `lidar_blind` zeros the depth. Because the backbone still receives a valid 4-channel tensor,
the forward **never crashes** — it just loses one source. Here we zero the depth channel
(`lidar_blind`) and re-run: the 4th channel goes flat, and the heatmap loses its lidar-driven cue.


In [ ]:
early_input_blind = early_input.clone()
early_input_blind[:, 3:4] = 0.0                      # lidar_blind: zero the depth channel
print("depth channel after lidar_blind: min=%.3f max=%.3f  (was [%.3f, %.3f])" %
      (early_input_blind[0,3].min(), early_input_blind[0,3].max(),
       early_input[0,3].min(), early_input[0,3].max()))
with torch.no_grad():
    hm_ok  = early_head(early_backbone(early_input)["16"])["heat"][0,0].sigmoid()
    hm_bl  = early_head(early_backbone(early_input_blind)["16"])["heat"][0,0].sigmoid()
print(f"heatmap peak: normal={hm_ok.max():.3f}  lidar_blind={hm_bl.max():.3f}")
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
ax[0].imshow(hm_ok.numpy(), cmap="hot"); ax[0].set_title("normal")
ax[1].imshow(hm_bl.numpy(), cmap="hot"); ax[1].set_title("lidar_blind (depth zeroed)")
plt.suptitle("Early fusion degrades gracefully: forward still runs, cue weakens"); plt.tight_layout(); plt.show()


## 3.2 · Mid fusion (feature-level)

Combine the **learned features**, not the raw data. Each sensor gets its own encoder to a feature
grid; the grids are **spatially aligned** (same 24×80, same cell (r,c) = same image region); then
they are concatenated channel-wise and squeezed back with a **1×1 conv**.

```
RGB (B,3,H,W) -> image backbone   -> F_cam (B,C, 24,80)
pts (velo)     -> image-grid encoder -> F_lid (B,Cl,24,80)   # aligned to F_cam
                       cat([F_cam, F_lid], dim=1) -> (B, C+Cl, 24,80) --Conv1x1--> F_fused (B,C,24,80)
                                                              -> head -> boxes
```
This is `MidFusion2D.forward` in `fusion/mid_2d/model.py`. The LiDAR encoder is
`LidarImageGridEncoder` in `common/backbones/lidar_encoder.py`.


In [ ]:
# ---- LidarImageGridEncoder: faithful copy of common/backbones/lidar_encoder.py ----
class PointMLP(nn.Module):
    def __init__(self, in_dim, out_dim, hidden=64):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(in_dim, hidden), nn.ReLU(inplace=True),
                                 nn.Linear(hidden, out_dim), nn.ReLU(inplace=True))
    def forward(self, x): return self.net(x)

class LidarImageGridEncoder(nn.Module):
    """points (velo) -> (B, Cl, Hg, Wg) aligned to the camera feature grid at stride s."""
    def __init__(self, out_channels=16, stride=16):
        super().__init__()
        self.stride = stride; self.out_channels = out_channels
        self.mlp = PointMLP(in_dim=4, out_dim=out_channels)   # (cam_x, cam_y, cam_z, intensity)

    def forward(self, points_list, calib_list, H, W, device):
        s = self.stride; Hg, Wg = H//s, W//s; B = len(points_list)
        out = torch.zeros(B, self.out_channels, Hg, Wg, device=device)
        for b in range(B):
            pts = points_list[b].to(device).float()
            if pts.shape[0] == 0: continue
            m = calib_list[b].torch_matrices(device, pts.dtype)
            n = pts.shape[0]
            h = torch.cat([pts[:, :3], torch.ones((n, 1), device=device, dtype=pts.dtype)], dim=1)
            cam = (m["R0"] @ m["V2C"] @ h.t()).t()[:, :3]      # (N,3) cam frame
            inten = pts[:, 3] if pts.shape[1] >= 4 else torch.zeros(n, device=device, dtype=pts.dtype)
            # project to image grid cells
            img = (m["P"] @ torch.cat([cam, torch.ones((n,1), device=device, dtype=pts.dtype)], dim=1).t()).t()[:, :3]
            depth = img[:, 2]
            uv = img[:, :2] / torch.where(depth == 0, torch.full_like(depth, 1e-9), depth).unsqueeze(1)
            cell = (uv / float(s)).long()
            valid = (depth > 0) & (cell[:, 0] >= 0) & (cell[:, 0] < Wg) & (cell[:, 1] >= 0) & (cell[:, 1] < Hg)
            if valid.sum() == 0: continue
            feat_in = torch.cat([cam[valid], inten[valid].unsqueeze(-1)], dim=-1)  # (M,4)
            feats = self.mlp(feat_in).float()                                       # (M,Cl)
            r = cell[valid, 1].long().clamp(0, Hg-1); c = cell[valid, 0].long().clamp(0, Wg-1)
            lin = r*Wg + c
            grid = torch.zeros(self.out_channels, Hg*Wg, device=device)
            grid.scatter_reduce_(1, lin.unsqueeze(0).expand(self.out_channels, -1),
                                  feats.t(), reduce="amax", include_self=True)      # max-pool per cell
            out[b] = grid.view(self.out_channels, Hg, Wg)
        return out

lid_enc = LidarImageGridEncoder(out_channels=cfg["lidar_feat_channels"], stride=cfg["stride"])
pts_t = torch.from_numpy(points).float()
f_lid = lid_enc([pts_t], [calib], H, W, device)

# ---- walk the intermediate steps with numbers ----
with torch.no_grad():
    m = calib.torch_matrices(device, torch.float32)
    n = pts_t.shape[0]
    h = torch.cat([pts_t[:, :3], torch.ones((n,1))], dim=1)
    cam = (m["R0"] @ m["V2C"] @ h.t()).t()[:, :3]
    img = (m["P"] @ torch.cat([cam, torch.ones((n,1))], dim=1).t()).t()[:, :3]
    depth = img[:, 2]; uv = img[:, :2]/torch.where(depth==0, torch.full_like(depth,1e-9), depth).unsqueeze(1)
    cell = (uv.float()/cfg["stride"]).long()
    valid = (depth>0)&(cell[:,0]>=0)&(cell[:,0]<Wg)&(cell[:,1]>=0)&(cell[:,1]<Hg)
    M = int(valid.sum())
print(f"input points N            : {n}")
print(f"projected to cam frame     : cam {tuple(cam.shape)}")
print(f"projected to image pixels  : uv {tuple(uv.shape)}, depth {tuple(depth.shape)}")
print(f"assigned to grid cells     : cell {tuple(cell.shape)},  valid (in-frame) M = {M}")
print(f"per-point MLP input feat_in : ({M}, 4)  -> feats ({M}, {cfg['lidar_feat_channels']})")
print(f"max-pool per cell -> F_lid  : {tuple(f_lid.shape)}  <- (B, Cl, Hg, Wg)  ALIGNED to F_cam")
occ = (f_lid[0].abs().sum(0) > 0).sum().item()
print(f"occupied grid cells        : {occ} / {Hg*Wg}  ({100*occ/(Hg*Wg):.1f}%)")


In [ ]:
# ---- visualize a few LiDAR feature channels + the occupancy map ----
fig, ax = plt.subplots(1, 4, figsize=(14, 3))
occ = (f_lid[0].abs().sum(0) > 0).float().detach().numpy()
ax[0].imshow(occ, cmap="gray"); ax[0].set_title("occupancy\n(non-empty cells)")
for i in range(3):
    ax[i+1].imshow(f_lid[0, i].detach().numpy(), cmap="magma")
    ax[i+1].set_title(f"F_lid channel {i}")
plt.suptitle("LiDAR image-grid features (24x80), aligned to the camera grid")
plt.tight_layout(); plt.show()


### Mid-fusion forward pass — the concat + 1×1 conv

The camera branch produces `F_cam (1, C, 24, 80)`; the LiDAR branch produces `F_lid (1, Cl, 24, 80)`.
Concatenating along channels gives `(1, C+Cl, 24, 80)` — **same height/width**, so cell (r,c) in
`F_cam` and cell (r,c) in `F_lid` describe the *same patch of the image*. The 1×1 conv mixes the
two channel-stacks per cell back down to `F_fused (1, C, 24, 80)`.


In [ ]:
torch.manual_seed(0)
img_backbone = TinyImageBackbone(in_channels=3, out_channels=cfg["fusion_channels"])
fuse_conv = nn.Conv2d(cfg["fusion_channels"] + cfg["lidar_feat_channels"], cfg["fusion_channels"], 1)
mid_head = CenterHead2D(cfg["fusion_channels"], cfg["num_classes"], cfg["head_hidden"])

f_cam = img_backbone(img_t)["16"]                  # (1,C,24,80)
print("F_cam  (image backbone) :", tuple(f_cam.shape))
print("F_lid  (lidar encoder)  :", tuple(f_lid.shape))
cat = torch.cat([f_cam, f_lid], dim=1)              # (1, C+Cl, 24, 80)
print("cat([F_cam, F_lid], dim=1):", tuple(cat.shape), " <- same Hg,Wg; channels stacked")
f_fused = fuse_conv(cat)                            # (1, C, 24, 80)
print("1x1 conv -> F_fused     :", tuple(f_fused.shape), " <- back to C channels, one mixed stream")
pred = mid_head(f_fused)
print("head -> heat/off/size   :", {k: tuple(v.shape) for k, v in pred.items()})


In [ ]:
# ---- visualize the alignment: channel-mean of each branch and the fused stream ----
with torch.no_grad():
    cam_mean = f_cam[0].mean(0).numpy()
    lid_mean = f_lid[0].abs().mean(0).numpy()
    fused_mean = f_fused[0].mean(0).numpy()
fig, ax = plt.subplots(1, 3, figsize=(13, 3.5))
for a, im, ttl in zip(ax, [cam_mean, lid_mean, fused_mean],
                      ["F_cam (cam branch)", "F_lid (lidar branch)", "F_fused (after 1x1 conv)"]):
    a.imshow(im, cmap="viridis"); a.set_title(ttl); a.set_aspect("equal")
plt.suptitle("All three share the same 24x80 grid — fusion = per-cell channel mix")
plt.tight_layout(); plt.show()


In [ ]:
# ---- train mid fusion so a box appears, then decode ----
target = build_target_2d(torch.from_numpy(gt_box[None]).float(), torch.from_numpy(gt_label[None]), cfg, device)
opt = torch.optim.Adam(list(img_backbone.parameters()) + list(lid_enc.parameters())
                       + list(fuse_conv.parameters()) + list(mid_head.parameters()), lr=3e-3)
for step in range(1, 121):
    opt.zero_grad()
    fc = img_backbone(img_t)["16"]; fl = lid_enc([pts_t], [calib], H, W, device).to(fc.dtype)
    p = mid_head(fuse_conv(torch.cat([fc, fl], dim=1)))
    loss = compute_2d_loss(p, target); loss.backward(); opt.step()
    if step % 40 == 0 or step == 1:
        print(f"step {step:3d}  loss={loss.item():.4f}")
with torch.no_grad():
    dec = decode_boxes2d(mid_head(fuse_conv(torch.cat([img_backbone(img_t)['16'],
                          lid_enc([pts_t],[calib],H,W,device)],dim=1))), cfg["stride"], k=10, thresh=0.15)[0]
print("\nGT box     :", gt_box.round(1))
if dec["boxes"].shape[0]:
    print("decoded box :", dec["boxes"][0].numpy().round(1), " score=%.3f" % dec["scores"][0])
else:
    print("decoded box : (none above threshold)")


### Mid-fusion blind modes

`MidFusion2D` zeros the offending branch's feature map (still `(1, C, 24, 80)`, just all zeros),
so concat + 1×1 conv keep working. The fused grid loses one source but keeps the other's spatial
layout — graceful degradation, no shape changes.


In [ ]:
with torch.no_grad():
    f_cam_ok = img_backbone(img_t)["16"]
    f_lid_ok = lid_enc([pts_t], [calib], H, W, device).to(f_cam_ok.dtype)
    # lidar_blind: zero F_lid ; cam_blind: zero F_cam
    fused_ok  = fuse_conv(torch.cat([f_cam_ok, f_lid_ok], dim=1))
    fused_lidblind = fuse_conv(torch.cat([f_cam_ok, torch.zeros_like(f_lid_ok)], dim=1))
    fused_camblind  = fuse_conv(torch.cat([torch.zeros_like(f_cam_ok), f_lid_ok], dim=1))
print("normal       F_fused shape:", tuple(fused_ok.shape), " mean=%.4f"%fused_ok.mean())
print("lidar_blind  F_fused shape:", tuple(fused_lidblind.shape), " mean=%.4f"%fused_lidblind.mean())
print("cam_blind    F_fused shape:", tuple(fused_camblind.shape),  " mean=%.4f"%fused_camblind.mean())
print("-> shapes never change; the surviving branch still carries signal")


## 3.3 · Late fusion (decision-level)

Run **two independent detectors** and merge their **box lists**. The camera detector sees RGB;
the LiDAR detector sees a rendered 3-channel `[depth, height, intensity]` image (the lidar-only-2D
baseline). Merge (`fusion/late_2d/model.py::_merge`): associate boxes across the two lists by IoU,
average (or max) the matched confidences, keep unmatched boxes, then NMS the combined list.
```
RGB -> cam detector -> cam_boxes
render_lidar_3ch -> lid detector -> lid_boxes
        IoU associate -> fuse conf (mean|max) -> NMS -> merged boxes
```


In [ ]:
# ---- build the two detector inputs ----
lid_img_np = render_lidar_3ch_image(points, calib, H, W)        # (H,W,3) [depth,height,intensity]
lid_img_t = torch.from_numpy(lid_img_np).permute(2, 0, 1).unsqueeze(0)   # (1,3,H,W)
print("camera input (RGB)        :", tuple(img_t.shape),   " range [%.2f, %.2f]" % (img_t.min(), img_t.max()))
print("lidar  input (3ch render) :", tuple(lid_img_t.shape), " range [%.2f, %.2f]" % (lid_img_t.min(), lid_img_t.max()))

fig, ax = plt.subplots(1, 4, figsize=(14, 3))
ax[0].imshow(img); ax[0].set_title("cam input (RGB)"); ax[0].axis("off")
for i, (n, cm) in enumerate(zip(["depth", "height(z)", "intensity"], ["viridis", "cividis", "magma"])):
    ax[i+1].imshow(lid_img_np[..., i], cmap=cm); ax[i+1].set_title(f"lid ch {n}"); ax[i+1].axis("off")
plt.suptitle("Late fusion: two detectors see two different images of the same scene")
plt.tight_layout(); plt.show()


In [ ]:
# ---- two independent detectors, each = backbone(3ch) + head ----
torch.manual_seed(0)
cam_backbone, cam_head = TinyImageBackbone(3, cfg["fusion_channels"]), CenterHead2D(cfg["fusion_channels"], 1, cfg["head_hidden"])
lid_backbone, lid_head = TinyImageBackbone(3, cfg["fusion_channels"]), CenterHead2D(cfg["fusion_channels"], 1, cfg["head_hidden"])

with torch.no_grad():
    cam_pred = cam_head(cam_backbone(img_t)["16"])
    lid_pred = lid_head(lid_backbone(lid_img_t)["16"])
print("cam detector outputs:", {k: tuple(v.shape) for k, v in cam_pred.items()})
print("lid detector outputs:", {k: tuple(v.shape) for k, v in lid_pred.items()})

# train both detectors on the same GT
target = build_target_2d(torch.from_numpy(gt_box[None]).float(), torch.from_numpy(gt_label[None]), cfg, device)
opt = torch.optim.Adam(list(cam_backbone.parameters())+list(cam_head.parameters())+
                       list(lid_backbone.parameters())+list(lid_head.parameters()), lr=3e-3)
for step in range(1, 121):
    opt.zero_grad()
    loss = compute_2d_loss(cam_head(cam_backbone(img_t)["16"]), target) + \
           compute_2d_loss(lid_head(lid_backbone(lid_img_t)["16"]), target)
    loss.backward(); opt.step()
    if step % 40 == 0 or step == 1: print(f"step {step:3d}  total loss={loss.item():.4f}")

with torch.no_grad():
    cam_out = decode_boxes2d(cam_head(cam_backbone(img_t)["16"]), cfg["stride"], k=10, thresh=0.15)[0]
    lid_out = decode_boxes2d(lid_head(lid_backbone(lid_img_t)["16"]), cfg["stride"], k=10, thresh=0.15)[0]
print("\nGT box :", gt_box.round(1))
print("cam boxes:", cam_out["boxes"].numpy().round(1), "scores", cam_out["scores"].numpy().round(3))
print("lid boxes:", lid_out["boxes"].numpy().round(1), "scores", lid_out["scores"].numpy().round(3))


### The merge, step by step

This is the heart of late fusion. Given the two decoded lists, the merge does four things:
1. **IoU matrix** between every cam box and every lid box.
2. **Greedy associate**: for each cam box, take the highest-IoU unmatched lid box above the
   threshold (`merge_iou_thresh=0.5`).
3. **Fuse confidence** on matched pairs: `mean` or `max` of the two scores.
4. **NMS** the combined list (matched + both sets of unmatched) at the same threshold.

We run it on the real detector outputs above, then again on a clean hand-set example so every
number is readable.


In [ ]:
def merge(cam, lid, iou_thresh=0.5, mode="mean"):
    cb, cs = cam["boxes"].cpu().numpy(), cam["scores"].cpu().numpy()
    lb, ls = lid["boxes"].cpu().numpy(), lid["scores"].cpu().numpy()
    if cb.shape[0] == 0: return lid
    if lb.shape[0] == 0: return cam
    iou = iou2d(cb, lb)                                  # (Ncam, Nlid)
    print("  IoU matrix (cam x lid):\n", np.round(iou, 3))
    used = set(); boxes, scores = [], []
    for i in range(cb.shape[0]):
        jv = np.where(iou[i] > iou_thresh)[0]; matched = None
        for j in jv:
            if j not in used: used.add(j); matched = j; break
        if matched is None:
            print(f"  cam box {i} -> no match (kept as-is, s={cs[i]:.3f})")
            boxes.append(cb[i]); scores.append(cs[i])
        else:
            s = (cs[i]+ls[matched])/2 if mode == "mean" else max(cs[i], ls[matched])
            print(f"  cam box {i} <-> lid box {matched}  IoU={iou[i,matched]:.3f}  conf {mode}={s:.3f}")
            boxes.append(cb[i]); scores.append(s)
    for j in range(lb.shape[0]):
        if j not in used:
            print(f"  lid box {j} -> no match (kept as-is, s={ls[j]:.3f})")
            boxes.append(lb[j]); scores.append(ls[j])
    boxes = np.array(boxes); scores = np.array(scores)
    keep = nms2d(boxes, scores, iou_thresh)
    print(f"  NMS -> keep indices {keep.tolist()}  (of {len(boxes)} merged)")
    return {"boxes": torch.tensor(boxes[keep]), "scores": torch.tensor(scores[keep])}

print("=== merge on the real two detector outputs (mode=mean, thresh=0.5) ===")
merged = merge(cam_out, lid_out, cfg["merge_iou_thresh"], cfg["merge_conf_mode"])
print("\nmerged boxes:", merged["boxes"].numpy().round(1),
      "scores", merged["scores"].numpy().round(3))
print("GT box       :", gt_box.round(1))


### A clean, fully-numerical merge example

To make the IoU/merge/NMS mechanics obvious, here is a hand-set case: 3 camera boxes and 2 lidar
boxes. Boxes A↔L1 and B↔L2 overlap above 0.5 (so they merge); camera box C has no lidar partner
(kept as-is). Then NMS removes nothing (the three are spatially separate).


In [ ]:
cam_ex = {"boxes": torch.tensor([[100,100,300,200],   # A
                                    [400,100,600,200],   # B
                                    [700,100,900,200]], dtype=torch.float32),   # C
           "scores": torch.tensor([0.9, 0.7, 0.5])}
lid_ex = {"boxes": torch.tensor([[110,110,310,210],   # L1  ~ overlaps A
                                 [420,120,620,210]], dtype=torch.float32),      # L2  ~ overlaps B
           "scores": torch.tensor([0.8, 0.6])}

print("cam boxes: A(0.9) B(0.7) C(0.5)")
print("lid boxes: L1(0.8) L2(0.6)\n")
print("--- mode = mean ---")
m_mean = merge(cam_ex, lid_ex, 0.5, "mean")
print("merged:", m_mean["boxes"].numpy().tolist(), "scores", m_mean["scores"].numpy().round(3))
print()
print("--- mode = max ---")
m_max = merge(cam_ex, lid_ex, 0.5, "max")
print("merged:", m_max["boxes"].numpy().tolist(), "scores", m_max["scores"].numpy().round(3))
print()
print("mean keeps the agreement (0.9+0.8)/2=0.85; max keeps the boldest 0.9.")
print("Either way C survives (no lid partner) and the three separated boxes all pass NMS.")


## 3.4 · Putting the three side by side

| | Early | Mid | Late |
|---|---|---|---|
| **fusion point** | input (4-ch image) | feature grid (concat + 1×1) | decision (box list) |
| **#backbones** | 1 (4-ch) | 1 cam + 1 lidar encoder | 2 (cam + lid) |
| **#heads** | 1 | 1 | 2 |
| **what each sensor 'knows'** | nothing separate | own features, mixed late | full own detector |
| **export** | one 4-ch engine | one engine (scatter in C++) | two engines + C++ merge/NMS |
| **blind mode** | zero a channel | zero a branch's features | zero a detector's input |

Trade-off in one line: **the earlier you fuse, the more the sensors cooperate (and the cheaper the
deploy); the later you fuse, the more independent and modular each sensor is (and the heavier the
deploy).** Mid is the usual winner — cooperative features, single-engine export.


In [ ]:
# ---- robustness micro-demo: each variant still emits boxes when one sensor is killed ----
def n_decoded(b):
    return b["boxes"].shape[0] if isinstance(b, dict) else b["scores"].shape[0]

# Early: lidar_blind
e_ok  = decode_boxes2d(early_head(early_backbone(early_input)["16"]), cfg["stride"], k=10, thresh=0.15)[0]
e_bl  = decode_boxes2d(early_head(early_backbone(early_input_blind)["16"]), cfg["stride"], k=10, thresh=0.15)[0]
# Mid: lidar_blind
def mid_forward(cam_blk=False, lid_blk=False):
    with torch.no_grad():
        fc = img_backbone(img_t)["16"] if not cam_blk else torch.zeros(1, cfg["fusion_channels"], Hg, Wg)
        fl = lid_enc([pts_t], [calib], H, W, device) if not lid_blk else torch.zeros(1, cfg["lidar_feat_channels"], Hg, Wg)
        return decode_boxes2d(mid_head(fuse_conv(torch.cat([fc, fl.to(fc.dtype)], dim=1))), cfg["stride"], k=10, thresh=0.15)[0]
m_ok = mid_forward(); m_bl = mid_forward(lid_blk=True)
# Late: lidar_blind (lidar detector sees zeros -> likely no box; camera still detects)
with torch.no_grad():
    l_ok = decode_boxes2d(cam_head(cam_backbone(img_t)["16"]), cfg["stride"], k=10, thresh=0.15)[0]
    l_bl_cam = decode_boxes2d(cam_head(cam_backbone(torch.zeros_like(img_t))["16"]), cfg["stride"], k=10, thresh=0.15)[0]

print("decoded box counts  (thresh=0.15):")
print(f"  Early  normal={n_decoded(e_ok)}   lidar_blind={n_decoded(e_bl)}")
print(f"  Mid    normal={n_decoded(m_ok)}   lidar_blind={n_decoded(m_bl)}")
print(f"  Late   cam_detector normal={n_decoded(l_ok)}   cam_blind={n_decoded(l_bl_cam)}")
print("\nNo variant crashes when a sensor is dropped — the surviving branch keeps producing boxes.")


## Recap

- **Early** = `cat([RGB, depth], dim=1)` -> 4-ch image -> one detector. Fusion *is* the input.
- **Mid** = `F_cam` + aligned `F_lid` -> `cat` + `1×1 conv` -> one fused grid -> one head. The
  sweet spot: cooperative features, single-engine export.
- **Late** = two detectors -> **IoU associate -> mean/max conf -> NMS**. Modular but heavier deploy.

Every number above came from a real forward pass on a synthetic KITTI-style frame, mirroring your
`fusion/early_2d`, `mid_2d`, `late_2d` models and the shared `heads.py` / `boxes2d.py` / `losses_2d.py`.

**Next up (Section 4):** the 3D / BEV variant — `lift-splat` (camera) + `PointPillars` (lidar) fused
in BEV, with the 3D head (`heat / off / height / size / yaw`) and the 3D IoU/NMS. Say the word and
I'll build that notebook next.
